In [1]:
import torch
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime


# Load my functions
from utils import minmax_normalise_tensor, midpoint_to_box, upscale_tensor, box_tensor_2_mid_points
from metrics import mse, rmse
from covariance import covariance_function, aggregate_columns_rows_subsetcase, predictive_mean, predictive_variance, predictive_distribution, predictive_distribution_cholesky, predictive_distribution_cholesky_correction

/Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: dlopen(/Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torchvision/image.so, 0x0006): Symbol not found: __ZN2at4_ops19empty_memory_format4callEN3c108ArrayRefIxEENS2_8optionalINS2_10ScalarTypeEEENS5_INS2_6LayoutEEENS5_INS2_6DeviceEEENS5_IbEENS5_INS2_12MemoryFormatEEE
  Referenced from: <67CD63CE-57E0-341F-B3B8-78729B03D2B3> /Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torchvision/image.so
  Expected in:     <18497461-1393-3DF8-BED0-DC986FDB1051> /Users/kimbente/opt/anaconda3/envs/py3.9/lib/python3.9/site-packages/torch/lib/libtorch_cpu.dylib
  warn(f"Failed to load image Python extension: {e}")


# Load scene

Scene target tensor: [N, C, H, W]  

with channels: 
- `[:, 0, :, :]` bed
- `[:, 1, :, :]` surface
- `[:, 2, :, :]` thickness
- `[:, 3, :, :]` mask
- `[:, 4, :, :]` firn
- `[:, 5, :, :]` errorbed

and polar stereographic coordinates
- `[:, 6, :, :]` y
- `[:, 7, :, :]` x

# Language/terminology
- low resolution input tensor
- target tensor
  - inferred
  - ground truth
- auxiliary tensor

# Easy case: 60 pixel HW images
- aligning grids: Aux grid is the same 
- perfect upscale: 1, 2, 3, 4, 5, 6 times upscale fix perfectly

In [2]:
scene_bed_tensor = torch.load('./torch_data/DOMEC_bed_scenes_60pixel.pt')
# scene_bed_tensor = torch.load('./torch_data/TRANSANT_bed_scenes_60pixel.pt')

print(scene_bed_tensor.shape)

torch.Size([400, 8, 60, 60])


In [14]:
def run_experiment(list_of_covariance_functions, scene_bed_tensor, n_scenes):

    # n_scenes (int): number of scenes in dataset to iterate over. 400 is the maximum here 

    # Define upscaling factor. We use 60 pixel images so that upscaling is not compromised for 2 to 6.
    up_factor_list = [2, 3, 4, 5, 6]
    # Generate list of column names
    column_names = [("up_") + str(x) for x in up_factor_list]

    # Empty dataframes for losses: rows represent scenes (to guide further investigation) and columns represent upscaling factors
    proposed_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = column_names)
    baseline_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = column_names)
    # NLL only makes sense for probabilistic method
    proposed_nll_df = pd.DataFrame(index = range(0, n_scenes), columns = column_names)

    # Log marginal likelihood placeholder
    proposed_lml_df = pd.DataFrame(index = range(0, n_scenes), columns = column_names)

    # for each variant of covariance function
    for c in list_of_covariance_functions:
        # for each upscaling_factor
        for u_index, u in enumerate(up_factor_list):
            # for each scene (convert to batches later)
            for i in range(0, n_scenes):
                # Normalise target bed topography scene and create explicit first dim, e.g. torch.Size([1, 60, 60])
                target_ground_truth = minmax_normalise_tensor(scene_bed_tensor[i, 0, :, :]).unsqueeze(0)
                # Extract target height-width: last dimension (since H == W)
                target_hw = target_ground_truth.shape[-1]
                lr_hw = int(target_hw / u)
                
                # Upscale (Increase scale of each pixel, reduce resolution) to generate low-res. input
                lr_bed = upscale_tensor(target_ground_truth, upscaling_factor = u)

                # Normalisation of high-resolution auxiliary channel to compute the base_covariance
                hr_aux = minmax_normalise_tensor(scene_bed_tensor[i, 1, :, :]).unsqueeze(0)

                ### Base covariance ###
                base_covariance = covariance_function(hr_aux)
                k_ah_al_tensor, k_al_al_tensor = aggregate_columns_rows_subsetcase(base_covariance = base_covariance, u = u)

                ### MEAN RECONSTRUCTION ###
                # hr_inferred = predictive_mean(lr_bed.unsqueeze(0), k_ah_al_tensor, k_al_al_tensor, noise = 0.05, mu = 0.5)
                # hr_covariance_inferred = predictive_variance(base_covariance, k_ah_al_tensor, k_al_al_tensor, noise = 0.05)

                hr_mean_inferred, hr_variance_inferred, lml = predictive_distribution_cholesky_correction(lr_bed.unsqueeze(0), base_covariance, k_ah_al_tensor, k_al_al_tensor, 
                                                                                noise = torch.tensor(0.05), mu = torch.tensor(0.5))

                # Save log marginal likelihood
                proposed_lml_df.iloc[i, u_index] = lml

                ### LOSS ###
                # inplace mutation of row i and column u (upscale_factor) of df
                proposed_rmse_df.iloc[i, u_index] = rmse(hr_mean_inferred, target_ground_truth).numpy().item()

                # NLL; store items not tensors in df
                proposed_nll_df.iloc[i, u_index] = torch.nn.functional.gaussian_nll_loss(hr_mean_inferred, target_ground_truth, hr_variance_inferred, full = False, eps = 1e-06, reduction = 'mean').numpy().item()

                ### BASELINE ###
                # For torch grid resample function: Normalised grid as image input [N, C, H, W] where N = 1 and C = 1. 
                # H_in and W_in are implicit: corners of midpoints are assumed to me -1, -1 (top left) and 1, 1 (bottom right).
                # Always first dim
                lr_input_grid = lr_bed.unsqueeze(0).unsqueeze(0)

                # Assuming boundries are the same for both: outer boundries are [-1, 1] for both hr and lr
                d = torch.tensor(np.linspace(start = (-1.0 + (2/target_hw)/2) , stop = (1.0 - (2/target_hw)/2), num = target_hw))
                meshx, meshy = torch.meshgrid((d, d), indexing = "xy")
                # x,y order
                target_grid = torch.stack((meshx, meshy), 2)
                target_grid = target_grid.unsqueeze(0) # add batch dim
                
                # Border works much better than zero: Since we sample at a higher resolution than the input we sample e.g. left of the leftmost HR location
                hr_bilinear = torch.nn.functional.grid_sample(lr_input_grid.float(), target_grid.float(), mode = 'bilinear', padding_mode = 'border', align_corners = False)
                hr_bilinear = hr_bilinear.squeeze()

                ### BASELINE LOSS ###
                baseline_rmse_df.iloc[i, u_index] = rmse(hr_bilinear, target_ground_truth.squeeze()).numpy().item()
    
    return(proposed_rmse_df, baseline_rmse_df, proposed_nll_df, proposed_lml_df)


In [15]:
list_of_covar_function = [covariance_function]
num_scenes = 10 # 400 is max

proposed_rmse_df, baseline_rmse_df, proposed_nll_df, proposed_lml_df = run_experiment(list_of_covariance_functions = list_of_covar_function, scene_bed_tensor = scene_bed_tensor, n_scenes = num_scenes)

ValueError: var is of incorrect size

In [19]:
hr_variance_inferred.shape

torch.Size([1, 60, 60])

In [13]:
def visualise_results(proposed_rmse_df, baseline_rmse_df, proposed_lml_df, n_scenes, domain_name):

    # RMSE
    fig = go.Figure()
    fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y =  baseline_rmse_df.mean(), mode = 'lines+markers', name = "Bilinear baseline"))
    fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_rmse_df.mean(), mode = 'lines+markers', name = "Proposed algorithm"))
    fig.update_layout(title = 'Reconstruction loss [RMSE] of proposed vs. baseline - {} scenes near domain {}'.format(n_scenes, domain_name))
    fig.update_xaxes(title_text = 'Upscaling factor')
    fig.update_yaxes(title_text = 'RMSE')
    fig.show()

    # LML
    fig = go.Figure()
    fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_lml_df.mean(), mode = 'lines+markers', name = "Proposed algorithm"))
    fig.update_layout(title = "Reconstruction Log marginal likelihood [LML] (larger is better)")
    fig.update_xaxes(title_text = 'Upscaling factor')
    fig.update_yaxes(title_text = 'LML')
    fig.show()

domain_name = "Dome C"
visualise_results(proposed_rmse_df, baseline_rmse_df, proposed_lml_df, num_scenes, domain_name)

In [8]:
"""
### LINE_BY_LINE VERSION ###

# Define upscaling factor
# 60 pixel images so that upscaling is easy for 0 to 6
up_factor_list = [2, 3, 4, 5, 6]
# up_factor_list = [5]
n_scenes = 400 # max 400, 200

# For loop first and wrap up into a function once it works
proposed_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6'])
baseline_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6'])
proposed_nll_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6'])

proposed_lml_df = pd.DataFrame(index = range(0, n_scenes), columns = ['up_2', 'up_3', 'up_4', 'up_5', 'up_6'])

# for each scene (or fist n_scenes)
for u_index, u in enumerate(up_factor_list):
    for i in range(0, n_scenes):
        # Normalise target bed topography scene and create explicit first dim, e.g. torch.Size([1, 60, 60])
        target_ground_truth = minmax_normalise_tensor(scene_bed_tensor[i, 0, :, :]).unsqueeze(0)
        # Extract target height-width: last dimension (since H == W)
        target_hw = target_ground_truth.shape[-1]
        lr_hw = int(target_hw / u)
        
        # Upscale (Increase scale of each pixel, reduce resolution) to generate low-res. input
        lr_bed = upscale_tensor(target_ground_truth, upscaling_factor = u)

        # Normalisation of high-resolution auxiliary channel to compute the base_covariance
        hr_aux = minmax_normalise_tensor(scene_bed_tensor[i, 1, :, :]).unsqueeze(0)

        ### Base covariance ###
        base_covariance = covariance_function(hr_aux)
        k_ah_al_tensor, k_al_al_tensor = aggregate_columns_rows_subsetcase(base_covariance = base_covariance, u = u)

        ### MEAN RECONSTRUCTION ###
        # hr_inferred = predictive_mean(lr_bed.unsqueeze(0), k_ah_al_tensor, k_al_al_tensor, noise = 0.05, mu = 0.5)
        # hr_covariance_inferred = predictive_variance(base_covariance, k_ah_al_tensor, k_al_al_tensor, noise = 0.05)

        hr_mean_inferred, hr_variance_inferred, lml = predictive_distribution_cholesky_correction(lr_bed.unsqueeze(0), base_covariance, k_ah_al_tensor, k_al_al_tensor, 
                                                                         noise = torch.tensor(0.05), mu = torch.tensor(0.5))

        # Save log marginal likelihood
        proposed_lml_df.iloc[i, u_index] = lml

        ### LOSS ###
        # inplace mutation of row i and column u (upscale_factor) of df
        proposed_rmse_df.iloc[i, u_index] = rmse(hr_mean_inferred, target_ground_truth).numpy().item()

        # NLL
        # torch.nn.functional.gaussian_nll_loss(input, target, var, full=False, eps=1e-06, reduction='mean')

        ### BASELINE ###
        # For torch grid resample function: Normalised grid as image input [N, C, H, W] where N = 1 and C = 1. 
        # H_in and W_in are implicit: corners of midpoints are assumed to me -1, -1 (top left) and 1, 1 (bottom right).
        # Always first dim
        lr_input_grid = lr_bed.unsqueeze(0).unsqueeze(0)

        # Assuming boundries are the same for both: outer boundries are [-1, 1] for both hr and lr
        d = torch.tensor(np.linspace(start = (-1.0 + (2/target_hw)/2) , stop = (1.0 - (2/target_hw)/2), num = target_hw))
        meshx, meshy = torch.meshgrid((d, d), indexing = "xy")
        # x,y order
        target_grid = torch.stack((meshx, meshy), 2)
        target_grid = target_grid.unsqueeze(0) # add batch dim
        
        # Border works much better than zero: Since we sample at a higher resolution than the input we sample e.g. left of the leftmost HR location
        hr_bilinear = torch.nn.functional.grid_sample(lr_input_grid.float(), target_grid.float(), mode = 'bilinear', padding_mode = 'border', align_corners = False)
        hr_bilinear = hr_bilinear.squeeze()

        ### BASELINE LOSS ###
        baseline_rmse_df.iloc[i, u_index] = rmse(hr_bilinear, target_ground_truth.squeeze()).numpy().item()
"""

'\n### LINE_BY_LINE VERSION ###\n\n# Define upscaling factor\n# 60 pixel images so that upscaling is easy for 0 to 6\nup_factor_list = [2, 3, 4, 5, 6]\n# up_factor_list = [5]\nn_scenes = 400 # max 400, 200\n\n# For loop first and wrap up into a function once it works\nproposed_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = [\'up_2\', \'up_3\', \'up_4\', \'up_5\', \'up_6\'])\nbaseline_rmse_df = pd.DataFrame(index = range(0, n_scenes), columns = [\'up_2\', \'up_3\', \'up_4\', \'up_5\', \'up_6\'])\nproposed_nll_df = pd.DataFrame(index = range(0, n_scenes), columns = [\'up_2\', \'up_3\', \'up_4\', \'up_5\', \'up_6\'])\n\nproposed_lml_df = pd.DataFrame(index = range(0, n_scenes), columns = [\'up_2\', \'up_3\', \'up_4\', \'up_5\', \'up_6\'])\n\n# for each scene (or fist n_scenes)\nfor u_index, u in enumerate(up_factor_list):\n    for i in range(0, n_scenes):\n        # Normalise target bed topography scene and create explicit first dim, e.g. torch.Size([1, 60, 60])\n        tar

- Add NLL loss
- Rewrite for batches
- Cholesky implementation for correction
- Vary the covariance function

In [ ]:
# Check covariance properies
# Hermitian: symmetry check
# print(k_al_al_tensor == k_al_al_tensor.mT)
# non-negative eigenvalue check
# torch.linalg.eigvalsh(k_al_al_tensor)

In [10]:
# LML mainly for HP opt.
fig = go.Figure()
fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y =  baseline_rmse_df.mean(), mode = 'lines+markers', name = "Bilinear baseline"))
fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_rmse_df.mean(), mode = 'lines+markers', name = "Bayesian Fusion using auxiliary surface data"))
fig.update_layout(title = "Reconstruction loss [RMSE] compared to baseline - 400 scenes near Dome C")
fig.update_xaxes(title_text = 'Upscaling factor')
fig.update_yaxes(title_text = 'RMSE')
fig.show()

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x = list(range(2, 6 + 1)), y = proposed_lml_df.mean(), mode = 'lines+markers', name = "Bayesian Fusion using auxiliary surface data"))
fig.update_layout(title = "Reconstruction Log marginal likelihood (larger is better)")
fig.update_xaxes(title_text = 'Upscaling factor')
fig.update_yaxes(title_text = 'LML')
fig.show()

- Determine size of scense based on HPs 

## Cholesky

In [ ]:
# Cholesky
a = torch.randn(3, 3)
# Returns a view of this tensor with the last two dimensions transposed. Thus this is suitable when first dims are batches etc.
a = a @ a.mT + 1e-3 # make symmetric positive-definite
print(a)
l = torch.linalg.cholesky(a)
print(l @ l.mT)

reg_inv = torch.linalg.inv(a)
print(reg_inv)

chol_inv = torch.cholesky_inverse(torch.linalg.cholesky(a))
print(chol_inv)

In [ ]:
a.shape

# Understanding torch.grid_sample() and torch.meshgrid()

- [Blog post about i,j (Matrix) and x,y (Cartesian) indexing](https://sparrow.dev/numpy-meshgrid/)
- [ptrblck pytorch discussion on grid_sample()](https://discuss.pytorch.org/t/solved-torch-grid-sample/51662)

Derive same grid from (a) ij Matrix indexing

In [ ]:
i = torch.tensor([1, 2, 3]) # called x but should be y
j = torch.tensor([4, 5, 6])

# if i == j order does not matter in input
ii, jj = torch.meshgrid(i, j, indexing = 'ij')

### i == y ###
# ii are the grid of all y-values (rows)
print(ii)

# The first row of y-values are on the same row thus have contants value
print(ii[0, :])

### j == x ###
print(jj)

# Change to x, y order
xy_grid = torch.stack((jj, ii), 2)

print(xy_grid)

Derive same grid from (b) xy Cartesian indexing. xy will be the default in futture pytorch versions.

In [ ]:
# Other way to derive at the same result
y = torch.tensor([1, 2, 3]) 
x = torch.tensor([4, 5, 6])

xx, yy = torch.meshgrid(x, y, indexing = 'xy')

xy_grid = torch.stack((xx, yy), 2)

print(xy_grid)

### Example from Patrick

In [ ]:
input = torch.arange(4*4).view(1, 1, 4, 4).float()
print(input)

# Create grid to upsample input
d = torch.linspace(-1, 1, 8)

### Using Cartesian xy indexing ###
# both grid axis are the same so order does not matter in this case.
xx, yy = torch.meshgrid((d, d), indexing = "xy")
# x,y order of last two dimensions of grid
grid = torch.stack((xx, yy), 2)
grid = grid.unsqueeze(0) # add batch dim

# 09/2023 default is align_corners = False
output = torch.nn.functional.grid_sample(input, grid, align_corners = True)
print(output)

### Using Matrix ij indexing ###
# This is the default and was used in the referance example I am trying to reporduce
# ii corresponds to rows in matrix indexing, so y-axis in Cartesian indexing
ii, jj = torch.meshgrid((d, d), indexing = "ij")
# Same: ii, jj = torch.meshgrid((d, d))
# Reverse order to match x, y Cartesian ordering: jj == xx, ii == yy in previous example
grid = torch.stack((jj, ii), 2)
grid = grid.unsqueeze(0) # add batch dim

# 09/2022 default is align_corners = False
output = torch.nn.functional.grid_sample(input, grid, align_corners = True)
print(output)

### Reproduce any grid with grid_sample()

From documentation:
"For example, values x = -1, y = -1 is the left-top pixel of input, and values x = 1, y = 1 is the right-bottom pixel of input." However these are the boarders of the pixels, not the midpoints. 

In [ ]:
dims = 8
random_values = torch.rand(size = (dims, dims)) * 100
print(random_values)

In [ ]:
# 2.0 is the range, devide by 2 because we have centroids (which are at the midpoint of each cell)
d = torch.tensor(np.linspace(start = (-1.0 + (2.0 / dims) / 2) , stop = (1.0 - (2.0 / dims) / 2), num = dims))
meshx, meshy = torch.meshgrid((d, d), indexing = "xy")
grid = torch.stack((meshx, meshy), 2)
grid = grid.unsqueeze(0) # add batch dim

torch.nn.functional.grid_sample(random_values.unsqueeze(0).unsqueeze(0).float(), grid.float())
# same as torch.nn.functional.grid_sample(random_values.unsqueeze(0).unsqueeze(0).float(), grid.float(), align_corners = False, mode = 'bilinear', padding_mode = 'border')
# Padding mode does not matter though
